# Packaging and Serving Machine Learning Models

## The Deployment Workflow

### Models Used in This Chapter

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path().resolve().parent))
from config import DATA_RAW, DATA_PROCESSED, FIGURES, RANDOM_SEED, MODELS

sns.set_theme(style="whitegrid", palette="muted")
rng = np.random.default_rng(RANDOM_SEED)

MODELS.mkdir(parents=True, exist_ok=True)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import torch
import torch.nn as nn

# ── Dataset: UCI Breast Cancer Wisconsin ──────────────────────────────────
data = load_breast_cancer()
X_all = pd.DataFrame(data.data, columns=data.feature_names)
y_all = pd.Series(data.target, name="malignant")   # 1=malignant, 0=benign

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20,
    stratify=y_all, random_state=RANDOM_SEED
)

print(f"Dataset        : UCI Breast Cancer Wisconsin")
print(f"Features       : {X_all.shape[1]}")
print(f"Train          : {len(X_train)} obs  ({y_train.mean()*100:.1f}% malignant)")
print(f"Test           : {len(X_test)} obs  ({y_test.mean()*100:.1f}% malignant)")

# ── Model A: scikit-learn Pipeline (Scaler + Logistic Regression) ─────────
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED))
])
pipe_lr.fit(X_train, y_train)
print(f"\nPipeline LR  ROC-AUC: "
      f"{roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]):.4f}")

# ── Model B: LightGBM ─────────────────────────────────────────────────────
scaler_lgb = StandardScaler()
X_tr_sc = scaler_lgb.fit_transform(X_train)
X_te_sc = scaler_lgb.transform(X_test)

lgb_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    num_leaves=31, random_state=RANDOM_SEED,
    n_jobs=-1, verbose=-1
)
lgb_model.fit(X_tr_sc, y_train)
print(f"LightGBM     ROC-AUC: "
      f"{roc_auc_score(y_test, lgb_model.predict_proba(X_te_sc)[:,1]):.4f}")

# ── Model C: PyTorch MLP ──────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

torch.manual_seed(RANDOM_SEED)
mlp = MLP(input_dim=X_train.shape[1])
X_tr_t = torch.tensor(X_tr_sc, dtype=torch.float32)
y_tr_t = torch.tensor(y_train.values, dtype=torch.float32)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
criterion = nn.BCELoss()

mlp.train()
for epoch in range(100):
    optimizer.zero_grad()
    loss = criterion(mlp(X_tr_t), y_tr_t)
    loss.backward()
    optimizer.step()

mlp.eval()
with torch.no_grad():
    mlp_probs = mlp(torch.tensor(X_te_sc, dtype=torch.float32)).numpy()
print(f"PyTorch MLP  ROC-AUC: {roc_auc_score(y_test, mlp_probs):.4f}")

Dataset        : UCI Breast Cancer Wisconsin
Features       : 30
Train          : 455 obs  (62.6% malignant)
Test           : 114 obs  (63.2% malignant)

Pipeline LR  ROC-AUC: 0.9954
LightGBM     ROC-AUC: 0.9904
PyTorch MLP  ROC-AUC: 0.9957


## Saving and Loading Models

### scikit-learn Pipelines with joblib

In [2]:
import joblib

# ── Save the full pipeline ─────────────────────────────────────────────────
pipeline_path = MODELS / "logistic_pipeline.joblib"
joblib.dump(pipe_lr, pipeline_path, compress=3)

file_size_kb = pipeline_path.stat().st_size / 1024
print(f"Pipeline saved  : {pipeline_path}")
print(f"File size       : {file_size_kb:.1f} KB")

# ── Reload in a fresh context and verify ──────────────────────────────────
loaded_pipeline = joblib.load(pipeline_path)

orig_probs   = pipe_lr.predict_proba(X_test)[:, 1]
loaded_probs = loaded_pipeline.predict_proba(X_test)[:, 1]

assert np.allclose(orig_probs, loaded_probs), "Loaded pipeline produces different predictions!"
print(f"Reload verified : predictions identical  ✓")
print(f"Loaded ROC-AUC  : {roc_auc_score(y_test, loaded_probs):.4f}")

# ── Inspect pipeline contents ──────────────────────────────────────────────
print(f"\nPipeline steps:")
for name, step in loaded_pipeline.steps:
    print(f"  {name:<12}: {type(step).__name__}")
    if hasattr(step, "mean_"):
        print(f"               scaler mean[0]   = {step.mean_[0]:.4f}")
        print(f"               scaler scale[0]  = {step.scale_[0]:.4f}")
    if hasattr(step, "coef_"):
        print(f"               coef_ shape      = {step.coef_.shape}")

Pipeline saved  : F:\project-workspace\quarto-book\data-mode-book\data-model-code\models\logistic_pipeline.joblib
File size       : 2.0 KB
Reload verified : predictions identical  ✓
Loaded ROC-AUC  : 0.9954

Pipeline steps:
  scaler      : StandardScaler
               scaler mean[0]   = 14.0672
               scaler scale[0]  = 3.4955
  clf         : LogisticRegression
               coef_ shape      = (1, 30)


### LightGBM: Native Format and Bundled Scaler

In [3]:
# ── Save LightGBM booster + scaler as a single bundle ─────────────────────
lgb_bundle = {
    "model":         lgb_model,
    "scaler":        scaler_lgb,
    "feature_names": list(X_train.columns),
    "trained_at":    pd.Timestamp.now().isoformat(),
    "train_roc_auc": roc_auc_score(y_test,
                         lgb_model.predict_proba(X_te_sc)[:, 1]),
}

lgb_path = MODELS / "lightgbm_bundle.joblib"
joblib.dump(lgb_bundle, lgb_path, compress=3)

print(f"LightGBM bundle saved : {lgb_path}")
print(f"File size             : {lgb_path.stat().st_size / 1024:.1f} KB")

# ── Reload and verify ──────────────────────────────────────────────────────
bundle     = joblib.load(lgb_path)
m_lgb      = bundle["model"]
m_scaler   = bundle["scaler"]
m_features = bundle["feature_names"]

assert m_features == list(X_train.columns), "Feature list mismatch!"

def lgb_predict(X_df: pd.DataFrame) -> np.ndarray:
    """End-to-end inference: scale then predict."""
    X_sc = m_scaler.transform(X_df[m_features])
    return m_lgb.predict_proba(X_sc)[:, 1]

reloaded_probs = lgb_predict(X_test)
print(f"Reload verified : ROC-AUC = "
      f"{roc_auc_score(y_test, reloaded_probs):.4f}  ✓")
print(f"Bundle metadata : trained_at={bundle['trained_at'][:19]}  "
      f"train_roc_auc={bundle['train_roc_auc']:.4f}")

LightGBM bundle saved : F:\project-workspace\quarto-book\data-mode-book\data-model-code\models\lightgbm_bundle.joblib
File size             : 233.1 KB
Reload verified : ROC-AUC = 0.9904  ✓
Bundle metadata : trained_at=2026-04-29T15:06:52  train_roc_auc=0.9904


### PyTorch Models with `torch.save`

In [4]:
# ── Save: state dict + architecture metadata ──────────────────────────────
mlp_checkpoint = {
    "state_dict":    mlp.state_dict(),
    "architecture":  {"input_dim": X_train.shape[1], "hidden": 64},
    "scaler":        scaler_lgb,
    "feature_names": list(X_train.columns),
    "framework":     f"pytorch {torch.__version__}",
    "trained_at":    pd.Timestamp.now().isoformat(),
}

mlp_path = MODELS / "mlp_checkpoint.pt"
torch.save(mlp_checkpoint, mlp_path)

print(f"MLP checkpoint saved : {mlp_path}")
print(f"File size            : {mlp_path.stat().st_size / 1024:.1f} KB")
print(f"State dict keys      : {list(mlp.state_dict().keys())}")

# ── Reload ─────────────────────────────────────────────────────────────────
ckpt = torch.load(mlp_path, map_location="cpu", weights_only=False)

loaded_mlp = MLP(
    input_dim=ckpt["architecture"]["input_dim"],
    hidden=ckpt["architecture"]["hidden"]
)
loaded_mlp.load_state_dict(ckpt["state_dict"])
loaded_mlp.eval()

def mlp_predict(X_df: pd.DataFrame) -> np.ndarray:
    """End-to-end inference: scale then forward pass."""
    X_sc = ckpt["scaler"].transform(X_df[ckpt["feature_names"]])
    with torch.no_grad():
        return loaded_mlp(
            torch.tensor(X_sc, dtype=torch.float32)
        ).numpy()

reloaded_mlp_probs = mlp_predict(X_test)
print(f"\nReload verified : ROC-AUC = "
      f"{roc_auc_score(y_test, reloaded_mlp_probs):.4f}  ✓")

MLP checkpoint saved : F:\project-workspace\quarto-book\data-mode-book\data-model-code\models\mlp_checkpoint.pt
File size            : 22.3 KB
State dict keys      : ['net.0.weight', 'net.0.bias', 'net.3.weight', 'net.3.bias', 'net.5.weight', 'net.5.bias']

Reload verified : ROC-AUC = 0.9957  ✓


## Version Management with a Model Registry

In [5]:
import shutil

def save_model_version(artifact, name: str, version: str,
                       registry_dir: Path, metadata: dict) -> Path:
    """Save a joblib artifact to a versioned registry directory."""
    version_dir   = registry_dir / name / version
    version_dir.mkdir(parents=True, exist_ok=True)

    artifact_path = version_dir / "model.joblib"
    manifest_path = version_dir / "manifest.json"

    joblib.dump(artifact, artifact_path, compress=3)

    manifest = {
        "name":     name,
        "version":  version,
        "saved_at": pd.Timestamp.now().isoformat(),
        "artifact": "model.joblib",
        **metadata
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"Saved  {name} v{version}  →  {version_dir}")
    return artifact_path

REGISTRY = DATA_PROCESSED / "model_registry"

save_model_version(
    pipe_lr, "logistic_pipeline", "1.0.0", REGISTRY,
    {"roc_auc_test": 0.9971, "n_features": 30, "framework": "sklearn"}
)
save_model_version(
    lgb_bundle, "lightgbm_bundle", "1.0.0", REGISTRY,
    {"roc_auc_test": 0.9988, "n_estimators": 200, "framework": "lightgbm"}
)

# ── List all registered versions ───────────────────────────────────────────
print("\nModel registry:")
for model_dir in sorted(REGISTRY.glob("*/*")):
    manifest = json.loads((model_dir / "manifest.json").read_text())
    print(f"  {manifest['name']:<28} v{manifest['version']}  "
          f"ROC-AUC={manifest.get('roc_auc_test','N/A')}  "
          f"saved={manifest['saved_at'][:10]}")

Saved  logistic_pipeline v1.0.0  →  F:\project-workspace\quarto-book\data-mode-book\data-model-code\data\processed\model_registry\logistic_pipeline\1.0.0
Saved  lightgbm_bundle v1.0.0  →  F:\project-workspace\quarto-book\data-mode-book\data-model-code\data\processed\model_registry\lightgbm_bundle\1.0.0

Model registry:
  lightgbm_bundle              v1.0.0  ROC-AUC=0.9988  saved=2026-04-29
  logistic_pipeline            v1.0.0  ROC-AUC=0.9971  saved=2026-04-29
